In [1]:
pip install torch transformers datasets

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 206.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 1.1 GB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 365.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 184.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.1/794.1 kB 194.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 166.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 195.1 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.4.5
    Uninstalling safetensors-0.4.5:
      Successfully uninstalled safetensors-0.4.5
  Attempting uninstall: regex
    Found existing installation: regex 2024.9.11
    Uninstalling regex-2024.9.11:
      Successfully uninstalled regex-2024.9.11
  Attempting uninstall: pyar

In [2]:
pip install torch transformers datasets


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import math
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer

CSV_PATH = "IMDB Dataset.csv"
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 5
LR = 3e-4
EMBED_DIM = 256
HEADS = 8
LAYERS = 4
FF_DIM = 1024

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

df = pd.read_csv("IMDB Dataset (1).csv")
texts = df["review"].astype(str).tolist()

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

enc = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="pt"
)

class IMDBDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        x = self.ids[idx]
        return x, x.clone()

dataset = IMDBDataset(enc["input_ids"])
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.heads = heads
        self.dk = d_model // heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B,T,C = x.shape
        q = self.q(x).view(B,T,self.heads,self.dk).transpose(1,2)
        k = self.k(x).view(B,T,self.heads,self.dk).transpose(1,2)
        v = self.v(x).view(B,T,self.heads,self.dk).transpose(1,2)

        scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.dk)
        mask = torch.triu(torch.ones(T,T,device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, -1e9)
        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1,2).contiguous().view(B,T,C)
        return self.o(out)

class FeedForward(nn.Module):
    def __init__(self,d_model,ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,ff),
            nn.GELU(),
            nn.Linear(ff,d_model)
        )
    def forward(self,x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self,d_model,heads,ff):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model)
        self.attn=MultiHeadSelfAttention(d_model,heads)
        self.ln2=nn.LayerNorm(d_model)
        self.ff=FeedForward(d_model,ff)
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        x=x+self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self,vocab,max_len,d_model,heads,layers,ff):
        super().__init__()
        self.tok=nn.Embedding(vocab,d_model)
        self.pos=nn.Embedding(max_len,d_model)
        self.blocks=nn.Sequential(*[
            Block(d_model,heads,ff) for _ in range(layers)
        ])
        self.ln=nn.LayerNorm(d_model)
        self.fc=nn.Linear(d_model,vocab)

    def forward(self,ids):
        B,T=ids.shape
        pos=torch.arange(T,device=ids.device).unsqueeze(0)
        x=self.tok(ids)+self.pos(pos)
        x=self.blocks(x)
        x=self.ln(x)
        return self.fc(x)

model = GPT(
    tokenizer.vocab_size,
    MAX_LEN,
    EMBED_DIM,
    HEADS,
    LAYERS,
    FF_DIM
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Training...")

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for ids, labels in loader:
        ids = ids.to(device)
        labels = labels.to(device)

        logits = model(ids)
        loss = criterion(
            logits.view(-1, tokenizer.vocab_size),
            labels.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: {total/len(loader):.4f}")

torch.save(model.state_dict(), "gpt_imdb.pth")
print("Model saved as gpt_imdb.pth")

def generate(prompt, max_new_tokens=40):
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            ids = ids[:, -MAX_LEN:]
            logits = model(ids)
            next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(0)
            ids = torch.cat([ids, next_token], dim=1)

    return tokenizer.decode(ids[0], skip_special_tokens=True)

while True:
    text = input("\nEnter prompt (exit to quit): ")
    if text.lower() == "exit":
        break
    print(generate(text))

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using: cuda


Training...
Epoch 1: 0.4572
Epoch 2: 0.0129


KeyboardInterrupt: 

In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# -----------------------------
# Dataset
# -----------------------------
dataset = load_dataset("stanfordnlp/imdb")
texts = dataset["train"]["text"][:5000]

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 128

enc = tokenizer(
    texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="pt"
)

class GPTDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        x = self.ids[idx]
        return {
            "input_ids": x[:-1],
            "labels": x[1:]
        }

train_loader = DataLoader(
    GPTDataset(enc["input_ids"]),
    batch_size=8,
    shuffle=True
)

# -----------------------------
# GPT Model
# -----------------------------
class GPTModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8,
                 layers=6, max_len=128, dropout=0.1):
        super().__init__()

        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers
        )

        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, ids):
        B, T = ids.shape

        pos = torch.arange(T, device=ids.device).unsqueeze(0)

        x = self.token_embed(ids) + self.pos_embed(pos)

        mask = torch.triu(
            torch.ones(T, T, device=ids.device),
            diagonal=1
        ).bool()

        x = self.transformer(
            x,
            mask=mask
        )

        x = self.ln(x)

        return self.fc(x)

model = GPTModel(tokenizer.vocab_size, max_len=MAX_LEN).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

EPOCHS = 5

print("Training...")

for epoch in range(EPOCHS):
    model.train()
    total = 0

    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(ids)

        loss = criterion(
            logits.reshape(-1, tokenizer.vocab_size),
            labels.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: {total/len(train_loader):.4f}")

torch.save(model.state_dict(), "mini_gpt_imdb.pth")
print("Model saved.")

# -----------------------------
# Generation
# -----------------------------
model.eval()

def generate(prompt, max_new_tokens=40, temperature=0.8, top_k=40):
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):

            if ids.size(1) >= MAX_LEN-1:
                break

            logits = model(ids)
            logits = logits[:, -1, :] / temperature

            values, indices = torch.topk(logits, top_k)

            probs = torch.softmax(values, dim=-1)

            next_id = indices.gather(
                -1,
                torch.multinomial(probs, 1)
            )

            ids = torch.cat([ids, next_id], dim=1)

    return tokenizer.decode(ids[0], skip_special_tokens=True)

while True:
    text = input("\nPrompt: ")

    if text.lower() == "exit":
        break

    print("\nGenerated:\n")
    print(generate(text))

Using: cuda


Generating unsupervised split: 100%|███████████████████████████████████| 50000/50000 [00:00<00:00, 363411.28 examples/s]


Training...
Epoch 1: 6.6194
Epoch 2: 5.8069
Epoch 3: 5.4541
Epoch 4: 5.1563
Epoch 5: 4.8821
Model saved.



Prompt:  this movie was 



Generated:

this movie was  the story was very bad, but not just not one of them. It's a bunch of story of the worst movies ever made. I have ever seen.<br /><br />The story is



Prompt:  this movie was 



Generated:

this movie was <br /><br />"Hi Goldbergie's a few laughs at best movie ever made. The one had to be a good movie. The story was not funny (or, but the



Prompt:  this movie was a 



Generated:

this movie was a , it's not funny.<br /><br />I had never seen a lot of movies, but I am so disappointed and the cinematography that the good thing would have the actors, but I
